In [1]:
import pandas as pd
import numpy as np
import glob
import json

from tensorflow.keras.preprocessing.text import Tokenizer, tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

In [2]:
files = glob.glob("dataset/*.csv")
print("Files:", files)

df_list = [pd.read_csv(file) for file in files]
df = pd.concat(df_list, ignore_index=True)

print("Initial shape:", df.shape)
print("Columns:", df.columns)

Files: ['dataset\\city_council_train.csv', 'dataset\\corporate_test1_df.csv', 'dataset\\corporate_test_df.csv', 'dataset\\corporate_train2_df.csv', 'dataset\\corporate_train_df.csv', 'dataset\\corporate_validation_df.csv', 'dataset\\housing_society_meetings.csv', 'dataset\\product_launch_meetings.csv', 'dataset\\test2_df.csv', 'dataset\\test_df.csv', 'dataset\\train2_df.csv', 'dataset\\train_df.csv', 'dataset\\validation2_df.csv', 'dataset\\validation_df.csv']
Initial shape: (24381, 12)
Columns: Index(['id', 'Item_UID', 'Meeting_UID', 'Transcript', 'Summary', 'Date',
       'meeting_id', 'Society', 'City', 'Meeting_Type', 'Company', 'Product'],
      dtype='object')


In [3]:
df.columns = df.columns.str.strip().str.lower()

# 🔴 CHANGE THIS BASED ON YOUR DATASET
df.rename(columns={
    'text': 'text',
    'summary': 'summary'
}, inplace=True)

In [4]:
# =========================
# CLEAN COLUMN NAMES
# =========================
df.columns = df.columns.str.strip().str.lower()

print("All columns:", df.columns.tolist())

# =========================
# AUTO DETECT TEXT COLUMN
# =========================
text_col = None
summary_col = None

for col in df.columns:
    if any(word in col for word in ['text', 'transcript', 'content', 'note']):
        text_col = col
    if any(word in col for word in ['summary', 'short', 'abstract']):
        summary_col = col

print("Detected text column:", text_col)
print("Detected summary column:", summary_col)

# =========================
# CHECK
# =========================
if text_col is None or summary_col is None:
    raise ValueError("❌ Could not detect text/summary columns. Check dataset manually.")

# =========================
# RENAME TO STANDARD
# =========================
df.rename(columns={
    text_col: 'text',
    summary_col: 'summary'
}, inplace=True)

print("Final columns:", df.columns.tolist())
print(df[['text','summary']].head())

All columns: ['id', 'item_uid', 'meeting_uid', 'transcript', 'summary', 'date', 'meeting_id', 'society', 'city', 'meeting_type', 'company', 'product']
Detected text column: transcript
Detected summary column: summary
Final columns: ['id', 'item_uid', 'meeting_uid', 'text', 'summary', 'date', 'meeting_id', 'society', 'city', 'meeting_type', 'company', 'product']
                                                text  \
0  Speaker 3: We have a proclamation this evening...   
1  Speaker 3: We now open the public hearing for ...   
2  Speaker 3: We have a proclamation this evening...   
3  Speaker 3: We have a proclamation this evening...   
4  Speaker 3: We have a proclamation this evening...   

                                             summary  
0  A proclamation recognizing Local Food Bank for...  
1  Approves a variance allowing a building height...  
2  A proclamation recognizing Veterans Support Ne...  
3  A proclamation recognizing Local Food Bank for...  
4  A proclamation welcom

In [5]:
# ❌ STOP if no data
if len(df) == 0:
    raise ValueError("Dataset is empty! Fix your CSV files.")

In [6]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['text'])

vocab_size = len(tokenizer.word_index) + 1

X = tokenizer.texts_to_sequences(df['text'])
y = tokenizer.texts_to_sequences(df['summary'])

In [7]:
max_len_text = 100
max_len_summary = 20

X = pad_sequences(X, maxlen=max_len_text, padding='post')
y = pad_sequences(y, maxlen=max_len_summary, padding='post')

In [8]:
decoder_input = y[:, :-1]
decoder_output = y[:, 1:]
decoder_output = np.expand_dims(decoder_output, -1)

print("X:", X.shape)
print("decoder_input:", decoder_input.shape)
print("decoder_output:", decoder_output.shape)

X: (24381, 100)
decoder_input: (24381, 19)
decoder_output: (24381, 19, 1)


In [9]:
encoder_inputs = Input(shape=(max_len_text,))
enc_emb = Embedding(vocab_size, 128)(encoder_inputs)

encoder_lstm = LSTM(256, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

decoder_inputs = Input(shape=(max_len_summary - 1,))
dec_emb = Embedding(vocab_size, 128)(decoder_inputs)

decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

decoder_dense = Dense(vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 100)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_1 (InputLayer)    │ (None, 19)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 100, 128)          │       9,178,624 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_1 (Embedding)       │ (None, 19, 128)           │       9,178,624 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ [(None, 256), (None,      │         394,240 │ embedding[0][0]            │
│                               │ 256), (None, 256)]        │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_1 (LSTM)                 │ [(None, 19, 256), (None,  │         394,240 │ embedding_1[0][0],         │
│                               │ 256), (None, 256)]        │                 │ lstm[0][1], lstm[0][2]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 19, 71708)         │      18,428,956 │ lstm_1[0][0]               │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 37,574,684 (143.34 MB)

 Trainable params: 37,574,684 (143.34 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.fit(
    [X, decoder_input],
    decoder_output,
    epochs=10,
    batch_size=8,
    steps_per_epoch=200   # ⬅️ force it
)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 146s 687ms/step - loss: 6.6982
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 110s 550ms/step - loss: 5.5514
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 415ms/step - loss: 4.6147
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 414ms/step - loss: 3.7205
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 415ms/step - loss: 2.8778
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 416ms/step - loss: 2.4387
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 413ms/step - loss: 2.2799
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 414ms/step - loss: 2.0755
Epoch 9/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 417ms/step - loss: 1.9623
Epoch 10/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 82s 412ms/step - loss: 1.8445


In [12]:
model.save("model/summarizer_model.keras")

In [13]:
tokenizer_json = tokenizer.to_json()
with open("model/tokenizer.json", "w") as f:
    f.write(tokenizer_json)

print("✅ Training complete!")

✅ Training complete!
